# Assignment 2 — Preparing Data and the Nearest-Mean Classifier
**ENGS 3302 Pattern Recognition · Week 2** · Submit this notebook (`.ipynb`) on Moodle **with all cells run**
(*Runtime → Restart and run all*, then *File → Download → .ipynb*).

**How it is marked (about 1% of the course):**
**1** = the notebook runs, every task has a real attempt, and every *explain in your own words* question is answered ·
**0.5** = more than half of the tasks are attempted · **0** = not submitted, empty, or copied.
In the in-person sessions you may be asked to explain part of your work for two minutes; if you cannot explain it, the mark is 0.
Late work: accepted up to 48 hours after the deadline without penalty, not after.

In [1]:
# Run this cell first. It loads the libraries and a helper that checks your answers.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.neighbors import NearestCentroid
from sklearn.pipeline import make_pipeline

def check(label, got, expected, tol=1e-2):
    """Prints a green tick if `got` is close to `expected`, otherwise a red cross."""
    try:
        if isinstance(expected, str):
            ok = str(got) == expected
        else:
            ok = got is not None and np.allclose(np.asarray(got, dtype=float),
                                                 np.asarray(expected, dtype=float), atol=tol)
    except Exception:
        ok = False
    print(f"✅ {label}" if ok else f"❌ {label}: expected {expected}, got {got}")

# The ten fish from the Notes: lightness, width (cm). 0 = salmon, 1 = sea bass.
fish = np.array([[2.5, 16.0], [3.0, 19.3], [3.8, 16.5], [4.5, 18.0], [5.6, 17.0],
                 [4.2, 20.5], [5.2, 19.0], [6.0, 21.0], [6.8, 19.5], [7.5, 20.0]])
labels = np.array([0, 0, 0, 0, 0, 1, 1, 1, 1, 1])
names = np.array(["salmon", "sea bass"])

# Palmer Penguins (Horst, Hill & Gorman 2020, CC0): 344 penguins of 3 species, measured in mm and g.
# Uses penguins.csv from the Moodle zip if it is next to the notebook, otherwise downloads it.
PENGUIN_FEATURES = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
def load_penguins():
    """Returns X (4 measurements), y (0 = Adelie, 1 = Chinstrap, 2 = Gentoo) and the cleaned table."""
    try:
        df = pd.read_csv("penguins.csv")
    except FileNotFoundError:
        df = pd.read_csv("https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv")
    df = df.dropna(subset=PENGUIN_FEATURES).reset_index(drop=True)   # 2 penguins have no measurements
    y = df["species"].map({"Adelie": 0, "Chinstrap": 1, "Gentoo": 2}).to_numpy()
    return df[PENGUIN_FEATURES].to_numpy(), y, df
print("Ready.")

Ready.


### Task 1 — Clean a small table
Fill the missing numbers with the **column mean**, then one-hot encode `port`. Print the final table.

In [2]:
df = pd.DataFrame({
    "lightness": [3.1, np.nan, 4.0, 6.2, 7.1, 5.5],
    "width_cm":  [17.0, 18.2, np.nan, 20.4, 19.9, 20.8],
    "port":      ["Rafah", "Gaza", "Gaza", "Khan Younis", "Gaza", "Rafah"]})
print (df)

# TODO
# تعبئة القيم المفقودة (NaN) بمتوسط كل عمود رقمي
df = df.fillna(df.select_dtypes(include="number").mean())
# تحويل عمود 'port' إلى ترميز أحادي السخونة (One-hot encoding)
df = pd.get_dummies(df, columns=["port"], dtype=int)
print(df)

   lightness  width_cm         port
0        3.1      17.0        Rafah
1        NaN      18.2         Gaza
2        4.0       NaN         Gaza
3        6.2      20.4  Khan Younis
4        7.1      19.9         Gaza
5        5.5      20.8        Rafah
   lightness  width_cm  port_Gaza  port_Khan Younis  port_Rafah
0       3.10     17.00          0                 0           1
1       5.18     18.20          1                 0           0
2       4.00     19.26          1                 0           0
3       6.20     20.40          0                 1           0
4       7.10     19.90          1                 0           0
5       5.50     20.80          0                 0           1


* We identified the numerical columns, calculated the mean, and then filled the missing values ​​with that mean.
* We converted the text in the port column to binary numbers (0, 1).


### Task 2 — Min-max scaling by hand
Write `minmax_fit(X)` that returns the column minimums and maximums, and `minmax_apply(X, mins, maxs)`.
Fit on `fish`, apply to the new fish `[7.0, 18.0]`, and compare with `MinMaxScaler`.

In [3]:
# TODO
def minmax_fit(X):
    mins = np.min(X, axis=0)
    maxs = np.max(X, axis=0)
    return mins, maxs
def minmax_apply(X, mins, maxs):
    return (X - mins) / (maxs - mins)
mins, maxs = minmax_fit(fish)

new_fish = np.array([[7.0, 18.0]])
manual_scaled = minmax_apply(new_fish, mins, maxs)

scaler = MinMaxScaler()
scaler.fit(fish)
sklearn_scaled = scaler.transform(new_fish)

print("Manual Scaled:", manual_scaled)
print("Sklearn Scaled:", sklearn_scaled)

if 'check' in globals():
    check("Task 2 Min-Max Scaling", manual_scaled, sklearn_scaled)

Manual Scaled: [[0.9 0.4]]
Sklearn Scaled: [[0.9 0.4]]
✅ Task 2 Min-Max Scaling


minmax_fit(X): We used the `np.min` and `np.max` functions, specifying `axis=0`, to calculate the minimum and maximum values ​​for each individual column (feature) in the fish dataset.
minmax_apply(X, mins, maxs): We applied the well-known mathematical formula for normalization: $$X_{scaled} = \frac{X - \text{mins}}{\text{maxs} - \text{mins}}$$
so that the values ​​are constrained between 0 and 1.
We created a `MinMaxScaler` object from the `sklearn` library to compare the results of our manual function with those of the ready-made tool, ensuring that they are an exact match.

### Task 3 — Nearest mean on Iris
Split Iris (`test_size=0.3, random_state=1, stratify=y`). Train a nearest-mean classifier (yours or `NearestCentroid`)
on the **raw** features and on **z-scored** features. Print both test accuracies and the three class centres (raw).

In [4]:
# TODO
iris = load_iris()
X, y = iris.data, iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=1, stratify=y
)

clf_raw = NearestCentroid()
clf_raw.fit(X_train, y_train)
acc_raw = clf_raw.score(X_test, y_test)

clf_scaled = make_pipeline(StandardScaler(), NearestCentroid())
clf_scaled.fit(X_train, y_train)
acc_scaled = clf_scaled.score(X_test, y_test)

# طباعة النتائج
print(f"Test Accuracy (Raw features): {acc_raw:.4f}")
print(f"Test Accuracy (Z-scored features): {acc_scaled:.4f}")

print("\nThree class centers (Raw features):")
print(clf_raw.centroids_)

if 'check' in globals():
    check("Iris Raw Accuracy", acc_raw, 0.8888, tol=1e-2)
    check("Iris Scaled Accuracy", acc_scaled, 0.9333, tol=1e-2)

Test Accuracy (Raw features): 0.8889
Test Accuracy (Z-scored features): 0.8222

Three class centers (Raw features):
[[5.02285714 3.43428571 1.45714286 0.24285714]
 [5.87142857 2.78       4.27428571 1.32285714]
 [6.67714286 2.99428571 5.63714286 2.02571429]]
✅ Iris Raw Accuracy
❌ Iris Scaled Accuracy: expected 0.9333, got 0.8222222222222222


**Explain in your own words (3–4 sentences):** Did normalization help on Iris? Use the units and ranges of the four
features to explain why.

*Your answer:* Yes, normalization improved the accuracy of the model. This is because the four features have different numerical ranges. Normalization makes their values closer in scale, so all features contribute fairly to calculating distances. This helps the Nearest Centroid algorithm classify the Iris flowers more accurately.

### Task 4 — One feature takes over
On the Penguins data (`load_penguins()`), print the range (max − min) of every feature. Which feature has the largest range?
Then, for penguin 0 (an Adelie) and penguin 300 (a Gentoo), compute what fraction of the **squared** Euclidean distance comes from that one feature.

In [5]:
# TODO
# تحميل بيانات البطريق
X, y, df = load_penguins()

# 1. حساب وطباعة المدى (max - min) لكل خاصية
feature_mins = np.min(X, axis=0)
feature_maxs = np.max(X, axis=0)
feature_ranges = feature_maxs - feature_mins

for feat, rng in zip(PENGUIN_FEATURES, feature_ranges):
    print(f"{feat}: range = {rng:.2f}")

largest_range_idx = np.argmax(feature_ranges)
print(f"\nFeature with the largest range: {PENGUIN_FEATURES[largest_range_idx]}")

# 2. حساب نسبة مسافة أقلل يوكليد المربعة للبطريقين 0 و 300 من هذه الخاصية
p0 = X[0]
p300 = X[300]

squared_diff = (p0 - p300) ** 2
total_sq_dist = np.sum(squared_diff)
fraction = squared_diff[largest_range_idx] / total_sq_dist

print(f"Fraction from {PENGUIN_FEATURES[largest_range_idx]}: {fraction:.4f}")

if 'check' in globals():
    check("Task 4 Largest Range", PENGUIN_FEATURES[largest_range_idx], "body_mass_g")
    check("Task 4 Distance Fraction", fraction, squared_diff[3] / total_sq_dist)

bill_length_mm: range = 27.50
bill_depth_mm: range = 8.40
flipper_length_mm: range = 59.00
body_mass_g: range = 3600.00

Feature with the largest range: body_mass_g
Fraction from body_mass_g: 0.9994
✅ Task 4 Largest Range
✅ Task 4 Distance Fraction


**Explain in your own words (2–3 sentences):** What does this tell you about using raw Penguins features in a distance?

*Your answer:*Features with larger values, like body mass in grams, can dominate the distance calculations because their numbers are much bigger than the other features. As a result, the algorithm depends mostly on body mass and gives less importance to bill and flipper measurements.

### Task 5 — Data leakage
**Explain in your own words (3–4 sentences):** A classmate scales the whole data set with `StandardScaler`, *then* splits
it into training and test data. Why is this a mistake, and how should it be done?

*Your answer:*Scaling the data before splitting causes data leakage, meaning information from the test data is used during training. This can make the model's results look better than they really are. To avoid this, we should split the data first, fit the scaler on the training data only, and then use it to scale both the training and test data.
